# Analog Holidays - 38h Offset Forecast + Cluster Filter

Thin orchestration notebook for running `AnalogSpecialDays` from the hourly wide holiday audit CSV.

This variant forecasts a 38-hour window that starts 14 hours before the holiday midnight, so the operational forecast is ready before the holiday begins.

The loading, normalization, forecasting, and plotting logic lives in `analog/analog_holidays.py`. This notebook only defines parameters and calls the plotting helpers.

The CSV export only preserves holiday flags, so this workflow targets holiday analogs and restricts the analog bank to the selector cluster `F/G/H` assigned to each target in `holiday_selector_features.csv`.

In [1]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import analog_holidays.analog.analog_special_days as analog_special_days_module
import analog_holidays.analog.analog_holidays as analog_holidays_module

analog_special_days_module = importlib.reload(analog_special_days_module)
analog_holidays_module = importlib.reload(analog_holidays_module)

from analog_holidays.analog.analog_holidays import (
    build_analog_ranking_table,
    build_run_summary,
    plot_analog_pair_sequences,
    plot_batch_inference_grid,
    plot_batch_pair_sequences_grid,
    plot_forecast_diagnostics,
    plot_ranked_analog_profiles,
    prepare_audit_working_copy,
    run_analog_holidays,
    run_analog_holidays_batch,
    tune_analog_holidays_optuna,
)

pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 20)

## Parameters

Adjust the series, target date, and analog hyperparameters for the holiday-only CSV source.

- SOURCE_PATH: path to the holiday CSV file consumed by the notebook.
- UNIQUE_ID: target series used to build analogs and generate the forecast.
- TARGET_DATE: holiday date whose operational forecast window should be estimated.
- FORECAST_START_OFFSET_HOURS: how many hours before the holiday midnight the forecast window starts.
- SEASON_LENGTH: hourly profile length to model, in hours. For this workflow it should match `FORECAST_START_OFFSET_HOURS + 24`.
- SPECIAL_LABELS: labels that define which days are treated as special candidates when selecting analogs.
- K: number of special neighbors kept after ranking X against Y by similarity; None uses every filtered candidate.
- TYPEDIST: metric used to rank holiday candidates against Y; supports pearson, euclidian, and dtw.
- TYPEREG: regressor type used in the analog reconstruction step.
- N_COMPONENTS: number of components for dimensionality-reduction methods such as PCR or PLS.
- LEVELS: prediction interval levels to compute for the forecast.
- MIN_SPECIAL_POINTS: minimum number of hours flagged as special inside a candidate block.
- MIN_EVENT_GAP: minimum separation between consecutive special events to avoid overly overlapping candidates.
- MAX_EVENTS: maximum number of special events used as the analog bank; None uses all available events.
- SELECTOR_FEATURES_PATH / CLUSTER_COLUMN / MATCH_TARGET_CLUSTER: selector-cluster filter that keeps only historical analogs in the same target cluster (`F`, `G`, or `H`) defined in `holiday_selector_features.csv`.
- MAX_PLOTTED_ANALOGS: maximum number of analogs shown in the comparative plots.

In [2]:
import shutil
from datetime import datetime

# Original CSV — never modified by this notebook.
_ORIGINAL_SOURCE = PROJECT_ROOT / 'analog_holidays' / 'holidays' / 'holiday_demand_mx.csv'

# Create a fresh timestamped working copy on every notebook launch.
_timestamp = datetime.now().strftime('%Y_%m_%d_%H_%M')
SOURCE_PATH = _ORIGINAL_SOURCE.with_name(f'holiday_demand_mx_{_timestamp}.csv')
shutil.copy2(_ORIGINAL_SOURCE, SOURCE_PATH)
print(f'Working copy: {SOURCE_PATH.name}')

UNIQUE_ID = 'SEN_demand_CEL'
UNIQUE_ID = 'SEN_demand_PEN'
UNIQUE_ID = 'SEN_demand_SIN'

Working copy: holiday_demand_mx_2026_05_25_11_57.csv


In [3]:
SPECIAL_LABELS = ('holiday',)
FORECAST_START_OFFSET_HOURS = 14
SEASON_LENGTH = 38  # 14 h pre-holiday + 24 h holiday
K = 1000
TYPEDIST = 'pearson'
TYPEREG = 'PCR'
N_COMPONENTS = 3
LEVELS = [80, 95]
MIN_SPECIAL_POINTS = 24  # require the 24 holiday hours inside each 38-h candidate window
MIN_EVENT_GAP = 24
MAX_EVENTS = None
MAX_PLOTTED_ANALOGS = 10

SELECTOR_FEATURES_PATH = PROJECT_ROOT / 'analog_holidays' / 'holidays' / 'holiday_selector_features.csv'
CLUSTER_COLUMN = 'analog_cluster'
MATCH_TARGET_CLUSTER = True

DATE_END = '2024-01-01'  # only dates strictly earlier than this cutoff are included in the study
OPTUNA_N_TRIALS = 25
OPTUNA_TIMEOUT_SEC = 900
OPTUNA_MAX_EVAL_DATES = 12
OPTUNA_RANDOM_SEED = 42

In [4]:
TARGET_DATES_2025 = [
    ('2025-01-01', "New Year's Day"),
    ('2025-02-03', 'Constitution Day'),
    ('2025-03-17', "Benito Juarez's Birthday"),
    ('2025-04-17', 'Maundy Thursday'),
    ('2025-04-18', 'Good Friday'),
    ('2025-04-19', 'Holy Saturday'),
    ('2025-05-01', 'Labor Day'),
    ('2025-09-16', 'Independence Day'),
    ('2025-11-17', 'Mexican Revolution Day'),
    ('2025-12-24', 'Christmas Eve'),
    ('2025-12-25', 'Christmas Day'),
    ('2025-12-31', "New Year's Eve"),
    # ===== 2026 =====
    ('2026-01-01', "New Year's Day"),    
    ('2026-02-02', 'Constitution Day'),
    ('2026-03-16', "Benito Juarez's Birthday"),
    ('2026-04-02', 'Maundy Thursday'),
    ('2026-04-03', 'Good Friday'),
    ('2026-04-04', 'Holy Saturday'),
    ('2026-05-01', 'Labor Day'),
    # ('2026-09-15', 'Independence Eve'),
    # ('2026-09-16', 'Independence Day'),
    # ('2026-11-16', 'Mexican Revolution Day'),
    # ('2026-12-24', 'Christmas Eve'),
    # ('2026-12-25', 'Christmas Day'),
    # ('2026-12-31', "New Year's Eve"),
]

TARGET_DATE = TARGET_DATES_2025[-1][0]

In [5]:
selector_features_df = pd.read_csv(SELECTOR_FEATURES_PATH, parse_dates=['date'])
selector_features_df['date'] = pd.to_datetime(selector_features_df['date']).dt.normalize()
target_ts = pd.Timestamp(TARGET_DATE).normalize()

target_cluster_df = selector_features_df.loc[
    selector_features_df['date'] == target_ts,
    ['date', 'holiday_name', 'holiday_day_type', CLUSTER_COLUMN],
]

if target_cluster_df.empty:
    raise ValueError(
        f'No selector cluster was found for TARGET_DATE={target_ts.date()} in {SELECTOR_FEATURES_PATH.name}.'
    )

TARGET_ANALOG_CLUSTER = target_cluster_df.iloc[0][CLUSTER_COLUMN]
eligible_cluster_analogs_df = selector_features_df.loc[
    (selector_features_df[CLUSTER_COLUMN] == TARGET_ANALOG_CLUSTER)
    & (selector_features_df['date'] < target_ts),
    ['date', 'holiday_name', 'anchor_holiday_name', 'holiday_day_type', CLUSTER_COLUMN],
] .sort_values('date').reset_index(drop=True)

print(
    f'TARGET_DATE={target_ts.date()} -> analog_cluster={TARGET_ANALOG_CLUSTER} '
    f'| eligible historical analog dates={len(eligible_cluster_analogs_df)}'
 )
display(target_cluster_df)
eligible_cluster_analogs_df.tail(20)

TARGET_DATE=2026-05-01 -> analog_cluster=F | eligible historical analog dates=13


,date,holiday_name,holiday_day_type,analog_cluster
80,2026-05-01,Labor Day,H2,F


,date,holiday_name,anchor_holiday_name,holiday_day_type,analog_cluster
0,2020-09-16,Independence Day,Independence Day,H2,F
1,2021-04-01,Maundy Thursday,Maundy Thursday,H1,F
2,2021-05-01,Labor Day,Labor Day,H2,F
3,2021-09-16,Independence Day,Independence Day,H2,F
4,2023-04-06,Maundy Thursday,Maundy Thursday,H1,F
5,2023-09-16,Independence Day,Independence Day,H2,F
6,2024-03-28,Maundy Thursday,Maundy Thursday,H1,F
7,2024-05-01,Labor Day,Labor Day,H2,F
8,2024-09-16,Independence Day,Independence Day,H2,F
9,2025-04-17,Maundy Thursday,Maundy Thursday,H1,F


In [6]:
rolling_target_items = [
    (pd.Timestamp(target_date).date().isoformat(), holiday_label)
    for target_date, holiday_label in TARGET_DATES_2025
]

selector_cluster_lookup = (
    selector_features_df
    .dropna(subset=[CLUSTER_COLUMN])
    .drop_duplicates(subset=['date'], keep='last')
    .set_index('date')[CLUSTER_COLUMN]
    .to_dict()
)

selector_anchor_lookup = (
    selector_features_df
    .dropna(subset=['anchor_holiday_name'])
    .drop_duplicates(subset=['date'], keep='last')
    .set_index('date')['anchor_holiday_name']
    .to_dict()
)

def _summary_param(summary_df, param_name, default=np.nan):
    matches = summary_df.loc[summary_df['param'] == param_name, 'value']
    return matches.iloc[0] if not matches.empty else default

rolling_optuna_results = {}
rolling_runs = {}
rolling_rows = []

for target_date, holiday_label in rolling_target_items:
    target_ts = pd.Timestamp(target_date).normalize()
    target_cluster = selector_cluster_lookup.get(target_ts, pd.NA)
    anchor_holiday_name = selector_anchor_lookup.get(target_ts, holiday_label)
    if pd.isna(anchor_holiday_name):
        anchor_holiday_name = holiday_label
    anchor_holiday_name = str(anchor_holiday_name)
    print(f'[{target_date}] tuning and forecasting [{anchor_holiday_name}]...')

    try:
        tuning_result = tune_analog_holidays_optuna(
            unique_id=UNIQUE_ID,
            source_path=SOURCE_PATH,
            train_end=target_ts,
            season_length=SEASON_LENGTH,
            forecast_start_offset_hours=FORECAST_START_OFFSET_HOURS,
            initial_k=K,
            initial_typedist=TYPEDIST,
            initial_typereg=TYPEREG,
            initial_n_components=N_COMPONENTS,
            n_trials=OPTUNA_N_TRIALS,
            timeout_sec=OPTUNA_TIMEOUT_SEC,
            max_eval_dates=OPTUNA_MAX_EVAL_DATES,
            random_seed=OPTUNA_RANDOM_SEED,
            special_labels=SPECIAL_LABELS,
            min_special_points=MIN_SPECIAL_POINTS,
            min_event_gap=MIN_EVENT_GAP,
            max_events=MAX_EVENTS,
            selector_features_path=SELECTOR_FEATURES_PATH,
            cluster_column=CLUSTER_COLUMN,
            match_target_cluster=MATCH_TARGET_CLUSTER,
        )
        rolling_optuna_results[target_date] = tuning_result

        best_config = tuning_result.best_config
        run = run_analog_holidays(
            unique_id=UNIQUE_ID,
            target_date=target_ts,
            source_path=SOURCE_PATH,
            season_length=SEASON_LENGTH,
            forecast_start_offset_hours=FORECAST_START_OFFSET_HOURS,
            k=int(best_config['k']),
            typedist=str(best_config['typedist']),
            typereg=str(best_config['typereg']),
            n_components=int(best_config['n_components']),
            levels=LEVELS,
            special_labels=SPECIAL_LABELS,
            min_special_points=MIN_SPECIAL_POINTS,
            min_event_gap=MIN_EVENT_GAP,
            max_events=MAX_EVENTS,
            expected_target_label=None,
            selector_features_path=SELECTOR_FEATURES_PATH,
            cluster_column=CLUSTER_COLUMN,
            match_target_cluster=MATCH_TARGET_CLUSTER,
        )
        rolling_runs[target_date] = run

        mae_window = np.nan
        mape_window_pct = np.nan
        if run.actual_profile is not None:
            mae_window = float(np.mean(np.abs(run.forecast_profile - run.actual_profile)))
            denom = np.where(np.abs(run.actual_profile) > 1e-9, np.abs(run.actual_profile), np.nan)
            ape_pct = np.abs(run.forecast_profile - run.actual_profile) / denom * 100.0
            if np.isfinite(ape_pct).any():
                mape_window_pct = float(np.nanmean(ape_pct))

        rolling_rows.append({
            'target_date': target_date,
            'holiday_label': holiday_label,
            'analog_cluster': target_cluster,
            'train_end': target_date,
            'eligible_tuning_dates': len(tuning_result.eligible_dates),
            'target_exists': run.target_exists,
            'target_has_complete_profile': run.target_has_complete_profile,
            'selected_analogs': len(run.positions),
            'fail': run.fail,
            'k': run.k,
            'typedist': run.typedist,
            'typereg': run.typereg,
            'n_components': run.n_components,
            'forecast_start': run.forecast_start,
            'forecast_end': run.forecast_end,
            'mae_window': mae_window,
            'mape_window_pct': mape_window_pct,
            'tuning_best_mean_mae': _summary_param(tuning_result.summary_df, 'best_mean_mae'),
            'tuning_best_mean_mape_pct': _summary_param(tuning_result.summary_df, 'best_mean_mape_pct'),
            'error': None,
        })
        print(
            f'[{target_date}] cluster={target_cluster} | eligible={len(tuning_result.eligible_dates)} | '
            f'k={run.k} | {run.typereg} | {run.typedist} | analogs={len(run.positions)} | '
            f'MAPE={mape_window_pct:.2f}%'
        )
    except Exception as exc:
        rolling_rows.append({
            'target_date': target_date,
            'holiday_label': holiday_label,
            'analog_cluster': target_cluster,
            'train_end': target_date,
            'eligible_tuning_dates': np.nan,
            'target_exists': False,
            'target_has_complete_profile': False,
            'selected_analogs': 0,
            'fail': True,
            'k': np.nan,
            'typedist': pd.NA,
            'typereg': pd.NA,
            'n_components': np.nan,
            'forecast_start': pd.NaT,
            'forecast_end': pd.NaT,
            'mae_window': np.nan,
            'mape_window_pct': np.nan,
            'tuning_best_mean_mae': np.nan,
            'tuning_best_mean_mape_pct': np.nan,
            'error': str(exc),
        })
        print(f'[{target_date}] ERROR: {exc}')

rolling_results_df = pd.DataFrame(rolling_rows)
rolling_metric_summary_df = rolling_results_df[['mae_window', 'mape_window_pct']].describe(include='all')
batch_result_2025 = analog_holidays_module.AnalogHolidayBatchResult(
    target_items=rolling_target_items,
    runs=rolling_runs,
    results_df=rolling_results_df,
    metric_summary_df=rolling_metric_summary_df,
)

optuna_result = rolling_optuna_results.get(TARGET_DATE)
if optuna_result is not None:
    K = int(optuna_result.best_config['k'])
    TYPEDIST = str(optuna_result.best_config['typedist'])
    TYPEREG = str(optuna_result.best_config['typereg'])
    N_COMPONENTS = int(optuna_result.best_config['n_components'])

rolling_daily_table = rolling_results_df[
    [
        'target_date', 'holiday_label', 'analog_cluster', 'eligible_tuning_dates',
        'k', 'typedist', 'typereg', 'n_components', 'selected_analogs',
        'mae_window', 'mape_window_pct', 'error',
    ]
] .copy()

print('Rolling nested tuning finished.')
display(rolling_daily_table)
rolling_metric_summary_df

[2025-01-01] tuning and forecasting [New Year's Day]...
[2025-01-01] cluster=G | eligible=12 | k=11 | PCR | pearson | analogs=11 | MAPE=1.77%
[2025-02-03] tuning and forecasting [Constitution Day]...
[2025-02-03] cluster=H | eligible=12 | k=12 | RidgeReg | pearson | analogs=12 | MAPE=2.59%
[2025-03-17] tuning and forecasting [Benito Juarez's Birthday]...
[2025-03-17] cluster=H | eligible=12 | k=13 | LassoReg | pearson | analogs=13 | MAPE=1.44%
[2025-04-17] tuning and forecasting [Maundy Thursday]...
[2025-04-17] cluster=F | eligible=12 | k=11 | LassoReg | pearson | analogs=9 | MAPE=4.87%
[2025-04-18] tuning and forecasting [Good Friday]...
[2025-04-18] cluster=H | eligible=12 | k=11 | LassoReg | pearson | analogs=11 | MAPE=1.86%
[2025-04-19] tuning and forecasting [Holy Saturday]...
[2025-04-19] cluster=H | eligible=12 | k=11 | LassoReg | pearson | analogs=11 | MAPE=5.37%
[2025-05-01] tuning and forecasting [Labor Day]...
[2025-05-01] cluster=F | eligible=12 | k=12 | RidgeReg | pearson

,target_date,holiday_label,analog_cluster,eligible_tuning_dates,k,typedist,typereg,n_components,selected_analogs,mae_window,mape_window_pct,error
0,2025-01-01,New Year's Day,G,12,11,pearson,PCR,11,11,462.969577,1.765228,None
1,2025-02-03,Constitution Day,H,12,12,pearson,RidgeReg,3,12,798.676127,2.587093,None
2,2025-03-17,Benito Juarez's Birthday,H,12,13,pearson,LassoReg,3,13,434.995344,1.441679,None
3,2025-04-17,Maundy Thursday,F,12,11,pearson,LassoReg,3,9,1849.316849,4.872246,None
4,2025-04-18,Good Friday,H,12,11,pearson,LassoReg,3,11,653.256513,1.860728,None
5,2025-04-19,Holy Saturday,H,12,11,pearson,LassoReg,3,11,1773.676904,5.370483,None
6,2025-05-01,Labor Day,F,12,12,pearson,RidgeReg,3,10,734.525836,1.949682,None
7,2025-09-16,Independence Day,F,12,10,pearson,PCR,2,10,3726.483270,9.994698,None
8,2025-11-17,Mexican Revolution Day,H,12,5,euclidian,PCR,2,5,1145.960815,3.426994,None
9,2025-12-24,Christmas Eve,H,12,4,euclidian,PCR,2,4,1423.497112,4.460572,None


,mae_window,mape_window_pct
count,19.000000,19.000000
mean,1183.950509,3.523061
std,851.692292,2.390456
min,434.995344,1.332522
25%,605.623047,1.905205
50%,841.871189,2.607906
75%,1497.490586,4.278936
max,3726.483270,9.994698


In [ ]:
import importlib.util

current_rolling_optuna_result = rolling_optuna_results.get(TARGET_DATE) if 'rolling_optuna_results' in globals() else None
current_cluster_row = rolling_daily_table.loc[rolling_daily_table['target_date'] == TARGET_DATE] if 'rolling_daily_table' in globals() else pd.DataFrame()
current_cluster = current_cluster_row['analog_cluster'].iloc[0] if not current_cluster_row.empty else pd.NA

optuna_typedist_choices = ['pearson', 'euclidian']
if importlib.util.find_spec('dtw') is not None:
    optuna_typedist_choices.append('dtw')

optuna_typereg_choices = ['PCR', 'PLS', 'RidgeReg', 'LassoReg', 'RF', 'Boosting']
optuna_k_upper_bound = 'min(number_of_special_days_in_training, 24)'
optuna_n_components_rule = 'integer in [2, min(k, season_length)] only when typereg is PCR or PLS'
optuna_dtw_rule = 'dtw_window_frac in [0.05, 1.0] only when typedist == dtw'

if current_rolling_optuna_result is None:
    rolling_daily_table.loc[rolling_daily_table['target_date'] == TARGET_DATE]
else:
    optuna_method_report = (
        f'OPTUNA METHOD REPORT FOR TARGET_DATE={TARGET_DATE}\n'
        f'- Optimization mode: single-objective minimization.\n'
        f'- Sampler: TPESampler(seed={OPTUNA_RANDOM_SEED}).\n'
        f'- Search budget: n_trials={OPTUNA_N_TRIALS}, timeout_sec={OPTUNA_TIMEOUT_SEC}.\n'
        f'- Rolling cutoff: train_end={TARGET_DATE}; only dates strictly earlier than the target are used for tuning.\n'
        f'- Backtest folds: {len(current_rolling_optuna_result.eligible_dates)} eligible historical holiday dates, capped by OPTUNA_MAX_EVAL_DATES={OPTUNA_MAX_EVAL_DATES}.\n'
        f'- Cluster restriction: match_target_cluster={MATCH_TARGET_CLUSTER}; target analog_cluster={current_cluster}.\n'
        f'- Objective function: minimize mean MAE across historical folds + 1000 * fail_rate.\n'
        f'- Search space for typedist: {optuna_typedist_choices}.\n'
        f'- Search space for typereg: {optuna_typereg_choices}.\n'
        f'- Search space for k: integer in [1, {optuna_k_upper_bound}].\n'
        f'- Conditional parameter for n_components: {optuna_n_components_rule}.\n'
        f'- Conditional DTW parameter: {optuna_dtw_rule}.\n'
        f'- Best configuration selected for this target: k={current_rolling_optuna_result.best_config["k"]}, '
        f'typedist={current_rolling_optuna_result.best_config["typedist"]}, '
        f'typereg={current_rolling_optuna_result.best_config["typereg"]}, '
        f'n_components={current_rolling_optuna_result.best_config["n_components"]}, '
        f'dtw_window={current_rolling_optuna_result.best_config["dtw_window"]}.'
    )
    print(optuna_method_report)
    display(current_rolling_optuna_result.summary_df)
    current_rolling_optuna_result.fold_metrics_df

In [ ]:
fig, axes = plot_batch_inference_grid(
    batch_result_2025,
    title=(
        f'Batch inference | {UNIQUE_ID}\n'
        f'Rolling nested tuning by target date | '
        f'window={SEASON_LENGTH}h | start=-{FORECAST_START_OFFSET_HOURS}h'
    ),
)
plt.show()

In [ ]:
current_rolling_optuna_result = rolling_optuna_results.get(TARGET_DATE) if 'rolling_optuna_results' in globals() else None
if current_rolling_optuna_result is not None:
    optuna_result = current_rolling_optuna_result
    K = int(optuna_result.best_config['k'])
    TYPEDIST = str(optuna_result.best_config['typedist'])
    TYPEREG = str(optuna_result.best_config['typereg'])
    N_COMPONENTS = int(optuna_result.best_config['n_components'])

run = batch_result_2025.runs.get(TARGET_DATE) if 'batch_result_2025' in globals() else None

if run is None:
    run = run_analog_holidays(
        unique_id=UNIQUE_ID,
        target_date=TARGET_DATE,
        source_path=SOURCE_PATH,
        season_length=SEASON_LENGTH,
        forecast_start_offset_hours=FORECAST_START_OFFSET_HOURS,
        k=K,
        typedist=TYPEDIST,
        typereg=TYPEREG,
        n_components=N_COMPONENTS,
        levels=LEVELS,
        special_labels=SPECIAL_LABELS,
        min_special_points=MIN_SPECIAL_POINTS,
        min_event_gap=MIN_EVENT_GAP,
        max_events=MAX_EVENTS,
        selector_features_path=SELECTOR_FEATURES_PATH,
        cluster_column=CLUSTER_COLUMN,
        match_target_cluster=MATCH_TARGET_CLUSTER,
    )

summary_df = build_run_summary(run)
print(summary_df.to_string(index=False))

In [ ]:
current_rolling_optuna_result = rolling_optuna_results.get(TARGET_DATE) if 'rolling_optuna_results' in globals() else None
if current_rolling_optuna_result is not None:
    optuna_result = current_rolling_optuna_result
    K = int(optuna_result.best_config['k'])
    TYPEDIST = str(optuna_result.best_config['typedist'])
    TYPEREG = str(optuna_result.best_config['typereg'])
    N_COMPONENTS = int(optuna_result.best_config['n_components'])

diagnostic_run = batch_result_2025.runs.get(TARGET_DATE) if 'batch_result_2025' in globals() else None

if diagnostic_run is None:
    diagnostic_run = run_analog_holidays(
        unique_id=UNIQUE_ID,
        target_date=TARGET_DATE,
        source_path=SOURCE_PATH,
        season_length=SEASON_LENGTH,
        forecast_start_offset_hours=FORECAST_START_OFFSET_HOURS,
        k=K,
        typedist=TYPEDIST,
        typereg=TYPEREG,
        n_components=N_COMPONENTS,
        levels=LEVELS,
        special_labels=SPECIAL_LABELS,
        min_special_points=MIN_SPECIAL_POINTS,
        min_event_gap=MIN_EVENT_GAP,
        max_events=MAX_EVENTS,
        expected_target_label=None,
        selector_features_path=SELECTOR_FEATURES_PATH,
        cluster_column=CLUSTER_COLUMN,
        match_target_cluster=MATCH_TARGET_CLUSTER,
    )

interval_rows = []
for lv in LEVELS:
    lo = diagnostic_run.interval_low.get(lv)
    hi = diagnostic_run.interval_high.get(lv)
    if lo is None or hi is None:
        continue
    width = hi - lo
    interval_rows.append({
        'level': lv,
        'average_interval_width': float(np.mean(width)),
        'max_interval_width': float(np.max(width)),
        'min_interval_width': float(np.min(width)),
    })

display(pd.DataFrame(interval_rows))

level_to_show = 95 if 95 in LEVELS else max(LEVELS)
hourly_interval_df = pd.DataFrame({
    'hour_relative_to_holiday': np.arange(len(diagnostic_run.forecast_profile)) - FORECAST_START_OFFSET_HOURS,
    'forecast_mean': diagnostic_run.forecast_profile,
    f'lower_limit_{level_to_show}': diagnostic_run.interval_low[level_to_show],
    f'upper_limit_{level_to_show}': diagnostic_run.interval_high[level_to_show],
})

hourly_interval_df.head(10)

### X/X2 and Y/Y2 Sequences — Batch Grid

Una gráfica por fecha pronosticada, mostrando los pares históricos X/X2 en azul claro y la secuencia Y/Y2 forecast en rojo.

En esta variante la ventana Y2/X2 tiene 38 horas y comienza 14 horas antes del inicio del holiday.

In [ ]:
fig_seq, axes_seq = plot_batch_pair_sequences_grid(
    batch_result_2025,
    title=(
        f'X/X2 y Y/Y2 por fecha pronosticada | {UNIQUE_ID}\n'
        f'Rolling nested tuning por fecha | '
        f'window={SEASON_LENGTH}h | start=-{FORECAST_START_OFFSET_HOURS}h'
    ),
)
plt.show()

In [ ]:
ranking_df = build_analog_ranking_table(run)
fig, axes = plot_ranked_analog_profiles(run, top_n=MAX_PLOTTED_ANALOGS)
plt.show()
ranking_df.head(MAX_PLOTTED_ANALOGS)